In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Crypto_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/bitcoin_1m.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20)

df.printSchema()

Raw Dataset Loaded
Total Rows: 1000
+-------------------+--------+--------+--------+--------+----------+
|          timestamp|    open|    high|     low|   close|    volume|
+-------------------+--------+--------+--------+--------+----------+
|2026-06-08 16:51:00|63423.33|63423.33| 63403.0|63421.65|0.46487861|
|2026-06-08 16:52:00|63411.48| 63419.3|63382.62|63382.62|1.63480953|
|2026-06-08 16:53:00|63382.63|63403.56|63368.54|63373.31|2.13842459|
|2026-06-08 16:54:00|63373.32|63390.64|63373.31|63383.07|0.36502369|
|2026-06-08 16:55:00| 63389.0|63412.56|63369.89|63412.56|0.55382608|
|2026-06-08 16:56:00|63403.99|63415.52| 63378.0|63403.78|0.48193165|
|2026-06-08 16:57:00|63407.56|63430.24| 63398.0|63407.98|0.34141393|
|2026-06-08 16:58:00|63404.66|63418.65|63384.48|63418.65|0.35754989|
|2026-06-08 16:59:00|63418.65| 63454.9|63418.65|63453.47|4.55834673|
|2026-06-08 17:00:00|63453.46| 63502.6|63439.73| 63502.6|1.81103019|
|2026-06-08 17:01:00|63509.15|63509.15| 63420.0| 63423.0|0.56430383

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+---------+----+----+---+-----+------+
|timestamp|open|high|low|close|volume|
+---------+----+----+---+-----+------+
|        0|   0|   0|  0|    0|     0|
+---------+----+----+---+-----+------+

Total Rows: 1000
Unique Rows: 1000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df = df.orderBy("timestamp")

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2026-06-08 16:51:00|2026-06-09 09:30:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("prev_time", F.lag("timestamp").over(w))

df = df.withColumn(
    "diff_min",
    (F.unix_timestamp("timestamp")
     - F.unix_timestamp("prev_time")) / 60
)

df.select(
    "timestamp",
    "prev_time",
    "diff_min"
).show(20, False)

gap_count = df.filter(F.col("diff_min") > 1).count()

print("Gap Count:", gap_count)

+-------------------+-------------------+--------+
|timestamp          |prev_time          |diff_min|
+-------------------+-------------------+--------+
|2026-06-08 16:51:00|NULL               |NULL    |
|2026-06-08 16:52:00|2026-06-08 16:51:00|1.0     |
|2026-06-08 16:53:00|2026-06-08 16:52:00|1.0     |
|2026-06-08 16:54:00|2026-06-08 16:53:00|1.0     |
|2026-06-08 16:55:00|2026-06-08 16:54:00|1.0     |
|2026-06-08 16:56:00|2026-06-08 16:55:00|1.0     |
|2026-06-08 16:57:00|2026-06-08 16:56:00|1.0     |
|2026-06-08 16:58:00|2026-06-08 16:57:00|1.0     |
|2026-06-08 16:59:00|2026-06-08 16:58:00|1.0     |
|2026-06-08 17:00:00|2026-06-08 16:59:00|1.0     |
|2026-06-08 17:01:00|2026-06-08 17:00:00|1.0     |
|2026-06-08 17:02:00|2026-06-08 17:01:00|1.0     |
|2026-06-08 17:03:00|2026-06-08 17:02:00|1.0     |
|2026-06-08 17:04:00|2026-06-08 17:03:00|1.0     |
|2026-06-08 17:05:00|2026-06-08 17:04:00|1.0     |
|2026-06-08 17:06:00|2026-06-08 17:05:00|1.0     |
|2026-06-08 17:07:00|2026-06-08

In [7]:
# =========================================
# MA10 & MA60
# =========================================

w10 = Window.orderBy("timestamp").rowsBetween(-9, 0)
w60 = Window.orderBy("timestamp").rowsBetween(-59, 0)

df = df.withColumn("MA10", F.avg("close").over(w10))
df = df.withColumn("MA60", F.avg("close").over(w60))

df.select(
    "timestamp",
    "close",
    "MA10",
    "MA60"
).show(20, False)

+-------------------+--------+------------------+------------------+
|timestamp          |close   |MA10              |MA60              |
+-------------------+--------+------------------+------------------+
|2026-06-08 16:52:00|63382.62|63382.62          |63382.62          |
|2026-06-08 16:53:00|63373.31|63377.965         |63377.965         |
|2026-06-08 16:54:00|63383.07|63379.666666666664|63379.666666666664|
|2026-06-08 16:55:00|63412.56|63387.89          |63387.89          |
|2026-06-08 16:56:00|63403.78|63391.06799999999 |63391.06799999999 |
|2026-06-08 16:57:00|63407.98|63393.88666666666 |63393.88666666666 |
|2026-06-08 16:58:00|63418.65|63397.42428571428 |63397.42428571428 |
|2026-06-08 16:59:00|63453.47|63404.42999999999 |63404.42999999999 |
|2026-06-08 17:00:00|63502.6 |63415.33777777777 |63415.33777777777 |
|2026-06-08 17:01:00|63423.0 |63416.10399999999 |63416.10399999999 |
|2026-06-08 17:02:00|63440.67|63421.90900000001 |63418.33727272727 |
|2026-06-08 17:03:00|63408.0 |6342

In [8]:
# =========================================
# ROC + MOMENTUM
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("close_lag10", F.lag("close", 10).over(w))

df = df.withColumn(
    "ROC",
    (F.col("close") - F.col("close_lag10"))
    / F.col("close_lag10") * 100
)

df = df.withColumn(
    "MOM",
    F.col("close") - F.col("close_lag10")
)

df.select(
    "close",
    "close_lag10",
    "ROC",
    "MOM"
).show(20, False)

+--------+-----------+---------------------+------------------+
|close   |close_lag10|ROC                  |MOM               |
+--------+-----------+---------------------+------------------+
|63382.62|NULL       |NULL                 |NULL              |
|63373.31|NULL       |NULL                 |NULL              |
|63383.07|NULL       |NULL                 |NULL              |
|63412.56|NULL       |NULL                 |NULL              |
|63403.78|NULL       |NULL                 |NULL              |
|63407.98|NULL       |NULL                 |NULL              |
|63418.65|NULL       |NULL                 |NULL              |
|63453.47|NULL       |NULL                 |NULL              |
|63502.6 |NULL       |NULL                 |NULL              |
|63423.0 |NULL       |NULL                 |NULL              |
|63440.67|63382.62   |0.09158662106425332  |58.049999999995634|
|63408.0 |63373.31   |0.054739132294024606 |34.69000000000233 |
|63396.11|63383.07   |0.0205733171334251

In [11]:
# =========================================
# RSI 14
# =========================================

w1 = Window.orderBy("timestamp")
w14 = Window.orderBy("timestamp").rowsBetween(-13, 0)

df = df.withColumn("change", F.col("close") - F.lag("close").over(w1))

df = df.withColumn(
    "gain",
    F.when(F.col("change") > 0, F.col("change")).otherwise(0)
)

df = df.withColumn(
    "loss",
    F.when(F.col("change") < 0, -F.col("change")).otherwise(0)
)

df = df.withColumn("avg_gain", F.avg("gain").over(w14))
df = df.withColumn("avg_loss", F.avg("loss").over(w14))

df = df.withColumn(
    "RS",
    F.when(F.col("avg_loss") == 0, None)
     .otherwise(F.col("avg_gain") / F.col("avg_loss"))
)

df = df.withColumn(
    "RSI",
    F.when(F.col("avg_loss") == 0, 100)
     .when(F.col("avg_gain") == 0, 0)
     .otherwise(
         100 - (100 / (1 + F.col("RS")))
     )
)

print("RSI Created")

df.select(
    "timestamp",
    "close",
    "change",
    "gain",
    "loss",
    "avg_gain",
    "avg_loss",
    "RS",
    "RSI"
).show(20, False)

RSI Created
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|timestamp          |close   |change             |gain              |loss              |avg_gain          |avg_loss          |RS                |RSI               |
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|2026-06-08 16:53:00|63373.31|NULL               |0.0               |0.0               |0.0               |0.0               |NULL              |100.0             |
|2026-06-08 16:54:00|63383.07|9.760000000002037  |9.760000000002037 |0.0               |4.880000000001019 |0.0               |NULL              |100.0             |
|2026-06-08 16:55:00|63412.56|29.489999999997963 |29.489999999997963|0.0               |13.083333333333334|0.0               |NULL              |100.0             

In [13]:
# =========================================
# STOCHASTIC OSCILLATOR
# =========================================

df = df.withColumn(
    "highest_high",
    F.max("high").over(w14)
)

df = df.withColumn(
    "lowest_low",
    F.min("low").over(w14)
)

df = df.withColumn(
    "stoch_k",
    F.when(
        (F.col("highest_high") - F.col("lowest_low")) == 0,
        None
    ).otherwise(
        (F.col("close") - F.col("lowest_low"))
        /
        (F.col("highest_high") - F.col("lowest_low"))
        * 100
    )
)

w3 = Window.orderBy("timestamp").rowsBetween(-2, 0)

df = df.withColumn(
    "stoch_d",
    F.avg("stoch_k").over(w3)
)

print("Stochastic Created")

df.select(
    "timestamp",
    "close",
    "highest_high",
    "lowest_low",
    "stoch_k",
    "stoch_d"
).show(20, False)

Stochastic Created
+-------------------+--------+------------+----------+------------------+------------------+
|timestamp          |close   |highest_high|lowest_low|stoch_k           |stoch_d           |
+-------------------+--------+------------+----------+------------------+------------------+
|2026-06-08 16:54:00|63383.07|63390.64    |63373.31  |56.31852279285086 |56.31852279285086 |
|2026-06-08 16:55:00|63412.56|63412.56    |63369.89  |100.0             |78.15926139642542 |
|2026-06-08 16:56:00|63403.78|63415.52    |63369.89  |74.27131273285418 |76.86327850856834 |
|2026-06-08 16:57:00|63407.98|63430.24    |63369.89  |63.11516155758857 |79.12882476348092 |
|2026-06-08 16:58:00|63418.65|63430.24    |63369.89  |80.79536039768553 |72.7272782293761  |
|2026-06-08 16:59:00|63453.47|63454.9     |63369.89  |98.31784495941623 |80.74278897156346 |
|2026-06-08 17:00:00|63502.6 |63502.6     |63369.89  |100.0             |93.03773511903393 |
|2026-06-08 17:01:00|63423.0 |63509.15    |63369.89

In [14]:
# =========================================
# BUY / SELL LABEL
# =========================================

df = df.withColumn(
    "label",
    F.when(F.col("MA10") > F.col("MA60"), 1).otherwise(0)
)

print("Buy/Sell Label Created")

print("Label Distribution")

df.groupBy("label").count().show()

df.select(
    "timestamp",
    "MA10",
    "MA60",
    "label"
).show(20, False)

Buy/Sell Label Created
Label Distribution
+-----+-----+
|label|count|
+-----+-----+
|    0|  549|
|    1|  451|
+-----+-----+

+-------------------+------------------+------------------+-----+
|timestamp          |MA10              |MA60              |label|
+-------------------+------------------+------------------+-----+
|2026-06-08 16:55:00|63412.56          |63412.56          |0    |
|2026-06-08 16:56:00|63408.17          |63408.17          |0    |
|2026-06-08 16:57:00|63408.10666666667 |63408.10666666667 |0    |
|2026-06-08 16:58:00|63410.7425        |63410.7425        |0    |
|2026-06-08 16:59:00|63419.288         |63419.288         |0    |
|2026-06-08 17:00:00|63433.17333333333 |63433.17333333333 |0    |
|2026-06-08 17:01:00|63431.719999999994|63431.719999999994|0    |
|2026-06-08 17:02:00|63432.838749999995|63432.838749999995|0    |
|2026-06-08 17:03:00|63430.078888888886|63430.078888888886|0    |
|2026-06-08 17:04:00|63426.68199999999 |63426.68199999999 |0    |
|2026-06-08 17:

In [15]:
# =========================================
# FEATURE TABLE
# =========================================

final_df = df.select(
    "timestamp",
    "open", "high", "low", "close", "volume",
    "MA10", "MA60",
    "ROC", "MOM",
    "RSI",
    "stoch_k", "stoch_d",
    "label"
)

print("Rows Before DropNA:", final_df.count())

final_df = final_df.dropna()

print("Rows After DropNA:", final_df.count())

Rows Before DropNA: 1000
Rows After DropNA: 990


In [18]:
# # =========================================
# # DELETE OLD BUCKET
# # =========================================

# import boto3
# from botocore.client import Config

# s3 = boto3.client(
#     "s3",
#     endpoint_url="http://minio:9000",
#     aws_access_key_id="admin",
#     aws_secret_access_key="password123",
#     config=Config(signature_version="s3v4")
# )

# bucket_name = "crypto-feature-table"

# # Xóa toàn bộ object trong bucket
# objects = s3.list_objects_v2(Bucket=bucket_name)

# if "Contents" in objects:
#     for obj in objects["Contents"]:
#         s3.delete_object(
#             Bucket=bucket_name,
#             Key=obj["Key"]
#         )

# # Xóa bucket
# s3.delete_bucket(Bucket=bucket_name)

# print("Bucket Deleted")

Bucket Deleted


In [19]:
# =========================================
# CREATE MINIO BUCKET
# =========================================

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123",
    config=Config(signature_version="s3v4")
)

bucket_name = "crypto-feature-table"

if bucket_name not in [
    b["Name"]
    for b in s3.list_buckets()["Buckets"]
]:
    s3.create_bucket(
        Bucket=bucket_name
    )
    print("Bucket Created")
else:
    print("Bucket Already Exists")

Bucket Created


In [20]:
# =========================================
# SAVE FEATURE TABLE
# =========================================

final_df.write \
    .mode("overwrite") \
    .parquet(
        "s3a://crypto-feature-table/features/"
    )

print("Feature Table Saved")

Feature Table Saved


In [23]:
# =========================================
# VERIFY OUTPUT
# =========================================

verify_df = spark.read.parquet(
    "s3a://crypto-feature-table/features/"
)

print(
    "Rows Written:",
    verify_df.count()
)

verify_df.show(30, False)

verify_df.printSchema()

Rows Written: 990
+-------------------+--------+--------+--------+--------+----------+------------------+------------------+--------------------+-------------------+------------------+--------------------+---------------------+-----+
|timestamp          |open    |high    |low     |close   |volume    |MA10              |MA60              |ROC                 |MOM                |RSI               |stoch_k             |stoch_d              |label|
+-------------------+--------+--------+--------+--------+----------+------------------+------------------+--------------------+-------------------+------------------+--------------------+---------------------+-----+
|2026-06-08 17:09:00|63389.93|63399.43|63354.48|63362.36|0.45940794|63417.210999999996|63420.50727272727 |-0.1435855281042953 |-91.11000000000058 |35.28822864524358 |10.630136986300839  |27.721968543884582   |0    |
|2026-06-08 17:10:00|63356.0 |63379.14|63341.28|63341.28|0.80578625|63401.079000000005|63413.905         |-0.254036842

In [25]:
# Chuyển file này sang .py
!jupyter nbconvert --to script DA.ipynb

[NbConvertApp] Converting notebook DA.ipynb to script
[NbConvertApp] Writing 8399 bytes to DA.py


In [33]:
!python DA.py

:: loading settings :: url = jar:file:/usr/local/spark-4.1.1-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c82f7126-8739-44ee-9344-88bdc00c0ae8;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 295ms :: artifacts dl 14ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.2 from central in [default]
	org

In [1]:
# import boto3

# s3 = boto3.client(
#     "s3",
#     endpoint_url="http://minio:9000",
#     aws_access_key_id="admin",
#     aws_secret_access_key="password123"
# )

# response = s3.list_objects_v2(
#     Bucket="crypto-raw-data"
# )

# for obj in response["Contents"]:
#     print(obj["Key"], obj["LastModified"])

altcoins_500d.csv 2026-06-06 14:58:03.066000+00:00
bitcoin_1m.csv 2026-06-10 04:38:06.392000+00:00
